# Consent-Tier Filtering Demo

This notebook demonstrates the moment where weeks of ethical reasoning become executable code.

In the voice dataset pipeline, every recording has metadata fields that encode governance decisions:
- `consent_tier` — who authorized what kind of use (`open`, `community`, `restricted`)
- `exclude_from_training` — a hard stop flag, regardless of tier
- `cultural_protocol` — free-text context that no schema can fully capture

When it's time to export recordings for model training, a single filter determines which recordings cross the boundary from archive to training data. This notebook shows you exactly what that filter does.

## 1. Load the sample metadata

The repository includes a small sample dataset with three recordings at different consent tiers. This is the same metadata used by the real export pipeline.

In [ ]:
import pandas as pd

df = pd.read_csv("../metadata/metadata_template.csv", dtype=str).fillna("")

print(f"Total recordings in archive: {len(df)}\n")
df[["file_id", "transcript", "consent_tier", "exclude_from_training", "cultural_protocol"]]

Notice:
- Recording **1** is `open`: approved for a public dataset
- Recording **2** is `community`: can train a community model, not for public release
- Recording **3** is `restricted`: with `exclude_from_training = true`, archival only, per elder council decision

All three recordings live in the same archive. The metadata fields are what separate them.

## 2. The filter

This is the exact filter from [05_export_ljspeech.ipynb](../notebooks/05_export_ljspeech.ipynb). Three conditions, one line of governance:

In [ ]:
eligible = df[
    (df["exclude_from_training"].str.lower() != "true") &
    (df["consent_tier"].str.lower().isin(["open", "community"])) &
    (df["transcript"].str.strip() != "")
]

excluded = df[~df.index.isin(eligible.index)]

print(f"Eligible for training: {len(eligible)}")
print(f"Excluded from training: {len(excluded)}")

## 3. What passed and what didn't

In [ ]:
print("ELIGIBLE for model training:")
print("=" * 50)
for _, row in eligible.iterrows():
    tier_label = "public dataset ok" if row["consent_tier"] == "open" else "community model only"
    print(f"  [{row['file_id']}] {row['transcript'][:60]}")
    print(f"       tier: {row['consent_tier']} ({tier_label})")
    print()

print("\nEXCLUDED from model training:")
print("=" * 50)
for _, row in excluded.iterrows():
    reason = row.get("exclude_reason", "") or f"consent_tier={row['consent_tier']}"
    print(f"  [{row['file_id']}] {row['transcript'][:60]}")
    print(f"       reason: {reason}")
    if row["cultural_protocol"]:
        print(f"       protocol: {row['cultural_protocol']}")
    print()

## 4. The distinction that matters after export

Both recordings 1 and 2 passed the filter. But they carry different obligations:

In [ ]:
print("Post-export obligations by consent tier:\n")

tier_obligations = {
    "open": [
        "Can be included in a publicly released model",
        "Can be shared in a public dataset",
        "Attribution requirements still apply",
    ],
    "community": [
        "Model trained on this data belongs to the community",
        "Model CANNOT be publicly released without community approval",
        "Dataset CANNOT be shared outside the community",
        "Community retains decision-making authority over all uses",
    ],
}

for _, row in eligible.iterrows():
    tier = row["consent_tier"].lower()
    print(f"Recording {row['file_id']} — tier: {tier}")
    for obligation in tier_obligations.get(tier, []):
        print(f"  - {obligation}")
    print()

## 5. What the excluded recording preserves

Recording 3 was excluded from training, but it was **not deleted**. It remains in the archive, accessible to community members, with its cultural context preserved in the metadata.

This is the key design principle: **preservation and training are separate decisions**. A recording can serve one purpose without serving the other.

In [ ]:
print("Archived but not trained on:\n")
for _, row in excluded.iterrows():
    print(f"Recording {row['file_id']}:")
    print(f"  Transcript: {row['transcript']}")
    print(f"  Consent tier: {row['consent_tier']}")
    print(f"  Cultural protocol: {row['cultural_protocol']}")
    print(f"  Exclude reason: {row['exclude_reason']}")
    print(f"  Knowledge keeper reviewed: {row['knowledge_keeper_reviewed']}")
    print()
    print("  This recording is preserved. It is not lost.")
    print("  It is simply not in the training pipeline — by design.")

---
## Discussion Questions

1. The filter is three lines of Python. Everything upstream such as community agreements, consent conversations, knowledge keeper reviews, and elder council decisions collapses into those three lines at export time. Is that a strength of this design (clarity, auditability) or a weakness (oversimplification)?

2. Recording 2 (`community` tier) passes the filter and enters the training dataset. But the obligation to keep the trained model within the community is *not enforced by code*, it's enforced by the agreement. What could go wrong? How would you add a technical safeguard?

3. Imagine a dataset with 10,000 recordings where 6,000 are `community`, 3,000 are `open`, and 1,000 are `restricted`. A researcher wants to train the best possible model. What tensions arise? How does the metadata system help navigate them?

4. Compare this approach to how large language models (GPT, Claude, Llama) handle training data consent. What is present here that is absent there? What is absent here that exists there?